In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib

import tensorflow as tf

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist


print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

        print("✓ GPU memory growth enabled")

    except RuntimeError as e:
        print("GPU memory-growth warning:", e)

TensorFlow: 2.20.0
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU memory growth enabled


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("✓ TensorFlow deterministic operations enabled")

except Exception as e:
    print("Determinism note:", e)

print("Seed:", SEED)

✓ TensorFlow deterministic operations enabled
Seed: 42


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1_DIR = (
    PROJECT_DIR /
    "phase1_segmentation"
)

P1A_DIR = (
    PHASE1_DIR /
    "phase1A_resolution"
)

P1B_DIR = (
    PHASE1_DIR /
    "phase1B_hyperparameters"
)

P1B_CONFIG_DIR = (
    P1B_DIR /
    "configs"
)

P1C_DIR = (
    PHASE1_DIR /
    "phase1C_capacity"
)

P1C_CONFIG_DIR = (
    P1C_DIR /
    "configs"
)

P1C_MODEL_DIR = (
    P1C_DIR /
    "models"
)

P1C_HISTORY_DIR = (
    P1C_DIR /
    "histories"
)

P1C_TABLE_DIR = (
    P1C_DIR /
    "tables"
)

P1C_PLOT_DIR = (
    P1C_DIR /
    "plots"
)


for folder in [
    P1C_CONFIG_DIR,
    P1C_MODEL_DIR,
    P1C_HISTORY_DIR,
    P1C_TABLE_DIR,
    P1C_PLOT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Phase 1C directory:")
print(P1C_DIR)

print("\n✓ Phase 1C folders ready.")

Phase 1C directory:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity

✓ Phase 1C folders ready.


In [ ]:
# ================================================================
# RESOLUTION
# ================================================================

resolution_path = (
    P1A_DIR /
    "configs" /
    "phase1A_final_resolution_decision.json"
)

assert resolution_path.exists()

with open(resolution_path, "r") as f:
    resolution_decision = json.load(f)

SELECTED_RESOLUTION = int(
    resolution_decision["selected_resolution"]
)


# ================================================================
# LEARNING RATE
# ================================================================

lr_path = (
    P1B_CONFIG_DIR /
    "phase1B1_learning_rate_decision.json"
)

assert lr_path.exists()

with open(lr_path, "r") as f:
    lr_decision = json.load(f)

SELECTED_LEARNING_RATE = float(
    lr_decision["selected_learning_rate"]
)


# ================================================================
# OPTIMIZER
# ================================================================

optimizer_path = (
    P1B_CONFIG_DIR /
    "phase1B2_optimizer_decision.json"
)

assert optimizer_path.exists()

with open(optimizer_path, "r") as f:
    optimizer_decision = json.load(f)

SELECTED_OPTIMIZER = (
    optimizer_decision["selected_optimizer"]
)


# ================================================================
# BATCH SIZE
# ================================================================

batch_path = (
    P1B_CONFIG_DIR /
    "phase1B3_batch_size_decision.json"
)

assert batch_path.exists()

with open(batch_path, "r") as f:
    batch_decision = json.load(f)

SELECTED_BATCH_SIZE = int(
    batch_decision["selected_batch_size"]
)


# ================================================================
# AUGMENTATION
# ================================================================

augmentation_path = (
    P1B_CONFIG_DIR /
    "phase1B4_augmentation_decision.json"
)

assert augmentation_path.exists()

with open(augmentation_path, "r") as f:
    augmentation_decision = json.load(f)

SELECTED_AUGMENTATION = (
    augmentation_decision[
        "selected_augmentation"
    ]
)


# ================================================================
# VERIFY
# ================================================================

assert SELECTED_RESOLUTION == 256
assert np.isclose(
    SELECTED_LEARNING_RATE,
    1e-4
)
assert SELECTED_OPTIMIZER == "Adam"
assert SELECTED_BATCH_SIZE == 2
assert SELECTED_AUGMENTATION is False


print("=" * 65)
print("INHERITED SETTINGS")
print("=" * 65)

print(
    f"Resolution   : "
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    f"Learning rate: "
    f"{SELECTED_LEARNING_RATE}"
)

print(
    f"Optimizer    : "
    f"{SELECTED_OPTIMIZER}"
)

print(
    f"Batch size   : "
    f"{SELECTED_BATCH_SIZE}"
)

print(
    "Augmentation : None"
)

print(
    "Loss         : BCE + Dice"
)

print("\n✓ Phase 1B decisions verified.")

INHERITED SETTINGS
Resolution   : 256×256
Learning rate: 0.0001
Optimizer    : Adam
Batch size   : 2
Augmentation : None
Loss         : BCE + Dice

✓ Phase 1B decisions verified.


In [ ]:
# CELL 6A — ONLY IF DATASET IS MISSING

import zipfile

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


if not DATASET_PATH.exists():

    assert OUTER_ZIP_PATH.exists(), (
        f"Archive not found: {OUTER_ZIP_PATH}"
    )

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as archive:

            archive.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")


    print("Extracting inner CAMUS archive...")

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )


assert DATASET_PATH.exists()

print("\n✓ CAMUS dataset ready:")
print(DATASET_PATH)

Extracting outer archive...
✓ Outer archive extracted.
Extracting inner CAMUS archive...

✓ CAMUS dataset ready:
/content/camus_data_final/database_nifti


In [ ]:
SPLIT_PATH = (
    PROJECT_DIR /
    "splits" /
    "patient_split_seed42.json"
)

assert SPLIT_PATH.exists()


with open(SPLIT_PATH, "r") as f:
    patient_split = json.load(f)


train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get(
        "validation",
        patient_split.get("val")
    )
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50

assert not (
    set(train_ids) & set(val_ids)
)

assert not (
    set(train_ids) & set(test_ids)
)

assert not (
    set(val_ids) & set(test_ids)
)


TRAIN_FRAMES = len(train_ids) * 2
VAL_FRAMES = len(val_ids) * 2


print("Train patients     :", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients      :", len(test_ids))

print("Training frames    :", TRAIN_FRAMES)
print("Validation frames  :", VAL_FRAMES)

print("\n✓ Split verified.")
print("✓ Test set excluded.")

Train patients     : 400
Validation patients: 50
Test patients      : 50
Training frames    : 800
Validation frames  : 100

✓ Split verified.
✓ Test set excluded.


In [ ]:
MAX_EPOCHS = 30
THRESHOLD = 0.50


CAPACITY_CONFIGS = {

    "small": [
        8,
        16,
        32,
        64,
        128
    ],

    "medium": [
        16,
        32,
        64,
        128,
        256
    ],

    "large": [
        32,
        64,
        128,
        256,
        512
    ]
}


print("=" * 70)
print("PHASE 1C — CONTROLLED MODEL CAPACITY EXPERIMENT")
print("=" * 70)


print("\nLOCKED")

print(
    "Resolution        :",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Learning rate     :",
    SELECTED_LEARNING_RATE
)

print(
    "Optimizer         :",
    SELECTED_OPTIMIZER
)

print(
    "Batch size        :",
    SELECTED_BATCH_SIZE
)

print(
    "Augmentation      : None"
)

print(
    "Loss              : BCE + Dice"
)

print(
    "Epoch budget      :",
    MAX_EPOCHS
)

print(
    "Threshold         :",
    THRESHOLD
)


print("\nONLY VARIABLE")
print("U-Net model capacity")


print("\nCAPACITY ARMS")

for name, filters in CAPACITY_CONFIGS.items():

    print(
        f"{name:>6}: {filters}"
    )


print("\nDATA")

print(
    "Training frames   :",
    TRAIN_FRAMES
)

print(
    "Validation frames :",
    VAL_FRAMES
)

print(
    "Test used         : NO"
)


print("\n✓ Phase 1C design frozen.")

PHASE 1C — CONTROLLED MODEL CAPACITY EXPERIMENT

LOCKED
Resolution        : 256×256
Learning rate     : 0.0001
Optimizer         : Adam
Batch size        : 2
Augmentation      : None
Loss              : BCE + Dice
Epoch budget      : 30
Threshold         : 0.5

ONLY VARIABLE
U-Net model capacity

CAPACITY ARMS
 small: [8, 16, 32, 64, 128]
medium: [16, 32, 64, 128, 256]
 large: [32, 64, 128, 256, 512]

DATA
Training frames   : 800
Validation frames : 100
Test used         : NO

✓ Phase 1C design frozen.


In [ ]:
def load_nifti_2d(file_path):

    nii = nib.load(
        str(file_path)
    )

    data = np.asarray(
        nii.get_fdata(),
        dtype=np.float32
    )

    if (
        data.ndim == 3
        and data.shape[-1] == 1
    ):
        data = data[:, :, 0]

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D image, got {data.shape}"
        )

    return data


def load_camus_frame(
    patient_id,
    phase
):

    patient_dir = (
        DATASET_PATH /
        patient_id
    )

    image = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}.nii.gz"
    )

    full_mask = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}_gt.nii.gz"
    )

    lv_mask = (
        full_mask == 1
    ).astype(np.float32)

    return image, lv_mask


def preprocess_frame(
    image,
    mask,
    resolution=SELECTED_RESOLUTION
):

    image_resized = cv2.resize(
        image,
        (
            resolution,
            resolution
        ),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask,
        (
            resolution,
            resolution
        ),
        interpolation=cv2.INTER_NEAREST
    )


    minimum = float(
        image_resized.min()
    )

    maximum = float(
        image_resized.max()
    )


    if maximum > minimum:

        image_resized = (
            image_resized - minimum
        ) / (
            maximum - minimum
        )

    else:

        image_resized = np.zeros_like(
            image_resized,
            dtype=np.float32
        )


    image_resized = (
        image_resized.astype(
            np.float32
        )
    )

    mask_resized = (
        mask_resized > 0.5
    ).astype(np.float32)


    return (
        image_resized[..., None],
        mask_resized[..., None]
    )


print("✓ Loader and preprocessing ready.")

✓ Loader and preprocessing ready.


In [ ]:
def frame_generator(
    patient_ids,
    shuffle=True,
    seed=SEED
):

    frame_list = [
        (pid, phase)
        for pid in patient_ids
        for phase in ["ED", "ES"]
    ]

    rng = np.random.default_rng(
        seed
    )


    while True:

        indices = np.arange(
            len(frame_list)
        )

        if shuffle:
            rng.shuffle(indices)


        for index in indices:

            pid, phase = (
                frame_list[index]
            )

            image, mask = (
                load_camus_frame(
                    pid,
                    phase
                )
            )

            x, y = preprocess_frame(
                image,
                mask
            )

            yield (
                x.astype(np.float32),
                y.astype(np.float32)
            )


OUTPUT_SIGNATURE = (

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    )
)


def build_datasets():

    training_dataset = (
        tf.data.Dataset
        .from_generator(
            lambda:
                frame_generator(
                    train_ids,
                    shuffle=True,
                    seed=SEED
                ),
            output_signature=
                OUTPUT_SIGNATURE
        )
        .batch(
            SELECTED_BATCH_SIZE,
            drop_remainder=True
        )
        .prefetch(1)
    )


    validation_dataset = (
        tf.data.Dataset
        .from_generator(
            lambda:
                frame_generator(
                    val_ids,
                    shuffle=False,
                    seed=SEED
                ),
            output_signature=
                OUTPUT_SIGNATURE
        )
        .batch(
            SELECTED_BATCH_SIZE,
            drop_remainder=False
        )
        .prefetch(1)
    )


    train_steps = (
        TRAIN_FRAMES //
        SELECTED_BATCH_SIZE
    )

    validation_steps = int(
        np.ceil(
            VAL_FRAMES /
            SELECTED_BATCH_SIZE
        )
    )


    return (
        training_dataset,
        validation_dataset,
        train_steps,
        validation_steps
    )


_, _, train_steps_check, val_steps_check = (
    build_datasets()
)

print(
    "Training steps/epoch:",
    train_steps_check
)

print(
    "Validation steps:",
    val_steps_check
)

print("\n✓ Dataset pipeline ready.")

Training steps/epoch: 400
Validation steps: 50

✓ Dataset pipeline ready.


In [ ]:
SMOOTH = 1e-6


def dice_coefficient(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )


    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )


    dice = (
        2.0 * intersection + SMOOTH
    ) / (
        denominator + SMOOTH
    )


    return tf.reduce_mean(
        dice
    )


def iou_metric(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )


    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
        -
        intersection
    )


    iou = (
        intersection + SMOOTH
    ) / (
        union + SMOOTH
    )


    return tf.reduce_mean(
        iou
    )


def dice_loss(
    y_true,
    y_pred
):

    return (
        1.0 -
        dice_coefficient(
            y_true,
            y_pred
        )
    )


def bce_dice_loss(
    y_true,
    y_pred
):

    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(
        bce
    )


    return (
        bce +
        dice_loss(
            y_true,
            y_pred
        )
    )


print("✓ Fixed BCE + Dice loss ready.")

✓ Fixed BCE + Dice loss ready.


In [ ]:
def conv_block(
    x,
    filters
):

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    return x


def build_capacity_unet(
    filters,
    input_shape=(256, 256, 1)
):

    assert len(filters) == 5

    f1, f2, f3, f4, f5 = filters


    inputs = tf.keras.Input(
        shape=input_shape
    )


    # ============================================================
    # ENCODER
    # ============================================================

    c1 = conv_block(
        inputs,
        f1
    )

    p1 = tf.keras.layers.MaxPooling2D()(
        c1
    )


    c2 = conv_block(
        p1,
        f2
    )

    p2 = tf.keras.layers.MaxPooling2D()(
        c2
    )


    c3 = conv_block(
        p2,
        f3
    )

    p3 = tf.keras.layers.MaxPooling2D()(
        c3
    )


    c4 = conv_block(
        p3,
        f4
    )

    p4 = tf.keras.layers.MaxPooling2D()(
        c4
    )


    # ============================================================
    # BOTTLENECK
    # ============================================================

    c5 = conv_block(
        p4,
        f5
    )


    # ============================================================
    # DECODER
    # ============================================================

    u6 = tf.keras.layers.Conv2DTranspose(
        f4,
        2,
        strides=2,
        padding="same"
    )(c5)

    u6 = tf.keras.layers.Concatenate()(
        [u6, c4]
    )

    c6 = conv_block(
        u6,
        f4
    )


    u7 = tf.keras.layers.Conv2DTranspose(
        f3,
        2,
        strides=2,
        padding="same"
    )(c6)

    u7 = tf.keras.layers.Concatenate()(
        [u7, c3]
    )

    c7 = conv_block(
        u7,
        f3
    )


    u8 = tf.keras.layers.Conv2DTranspose(
        f2,
        2,
        strides=2,
        padding="same"
    )(c7)

    u8 = tf.keras.layers.Concatenate()(
        [u8, c2]
    )

    c8 = conv_block(
        u8,
        f2
    )


    u9 = tf.keras.layers.Conv2DTranspose(
        f1,
        2,
        strides=2,
        padding="same"
    )(c8)

    u9 = tf.keras.layers.Concatenate()(
        [u9, c1]
    )

    c9 = conv_block(
        u9,
        f1
    )


    outputs = tf.keras.layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c9)


    segmentation_model = (
        tf.keras.Model(
            inputs,
            outputs
        )
    )


    return segmentation_model

In [ ]:
capacity_parameter_counts = {}


for capacity_name, filters in (
    CAPACITY_CONFIGS.items()
):

    tf.keras.backend.clear_session()
    gc.collect()

    tf.random.set_seed(SEED)

    capacity_model = (
        build_capacity_unet(
            filters=filters,
            input_shape=(
                SELECTED_RESOLUTION,
                SELECTED_RESOLUTION,
                1
            )
        )
    )

    parameter_count = int(
        capacity_model.count_params()
    )

    capacity_parameter_counts[
        capacity_name
    ] = parameter_count


    print(
        f"{capacity_name:>6} | "
        f"filters={filters} | "
        f"parameters={parameter_count:,}"
    )


    del capacity_model


tf.keras.backend.clear_session()
gc.collect()


# Current medium model MUST reproduce the previous architecture
assert (
    capacity_parameter_counts[
        "medium"
    ]
    ==
    1_940_817
)


print(
    "\n✓ Medium capacity exactly reproduces "
    "the Phase 1A/1B baseline architecture."
)

 small | filters=[8, 16, 32, 64, 128] | parameters=485,673
medium | filters=[16, 32, 64, 128, 256] | parameters=1,940,817
 large | filters=[32, 64, 128, 256, 512] | parameters=7,759,521

✓ Medium capacity exactly reproduces the Phase 1A/1B baseline architecture.


In [ ]:
capacity_protocol = {

    "experiment":
        "Phase 1C — Controlled Model Capacity Experiment",

    "status":
        "PROTOCOL FROZEN BEFORE TRAINING",

    "variable_tested":
        "U-Net filter capacity",

    "capacity_configurations":
        CAPACITY_CONFIGS,

    "parameter_counts":
        capacity_parameter_counts,

    "fixed_settings": {

        "resolution":
            SELECTED_RESOLUTION,

        "learning_rate":
            SELECTED_LEARNING_RATE,

        "optimizer":
            SELECTED_OPTIMIZER,

        "batch_size":
            SELECTED_BATCH_SIZE,

        "augmentation":
            False,

        "loss":
            "BCE + Dice",

        "epochs":
            MAX_EPOCHS,

        "threshold":
            THRESHOLD,

        "seed":
            SEED
    },

    "train_patients":
        len(train_ids),

    "validation_patients":
        len(val_ids),

    "train_frames_per_epoch":
        TRAIN_FRAMES,

    "validation_frames":
        VAL_FRAMES,

    "selection_dataset":
        "Validation set only",

    "test_used":
        False,

    "selection_metrics": [
        "overall Dice",
        "ED Dice",
        "ES Dice",
        "IoU",
        "normalized HD95",
        "ED area MAE",
        "ES area MAE",
        "overall area MAE",
        "training efficiency"
    ]
}


CAPACITY_PROTOCOL_PATH = (
    P1C_CONFIG_DIR /
    "phase1C_capacity_protocol.json"
)


with open(
    CAPACITY_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        capacity_protocol,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1C PROTOCOL FROZEN")
print("=" * 70)

for name in CAPACITY_CONFIGS:

    print(
        f"{name:>6}: "
        f"{capacity_parameter_counts[name]:,} parameters"
    )


print("\nTest used: NO")

print("\nSaved:")
print(CAPACITY_PROTOCOL_PATH)

print("\n✓ Capacity protocol frozen before training.")

PHASE 1C PROTOCOL FROZEN
 small: 485,673 parameters
medium: 1,940,817 parameters
 large: 7,759,521 parameters

Test used: NO

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/configs/phase1C_capacity_protocol.json

✓ Capacity protocol frozen before training.


In [ ]:
# ================================================================
# PHASE 1C — CELL 15
# CONTROLLED CAPACITY TRAINING FUNCTION
# ================================================================

def train_capacity_experiment(
    capacity_name
):

    assert capacity_name in CAPACITY_CONFIGS

    filters = CAPACITY_CONFIGS[
        capacity_name
    ]


    print("\n" + "=" * 70)
    print(
        f"TRAINING CAPACITY: "
        f"{capacity_name.upper()}"
    )
    print("=" * 70)

    print("Filters:", filters)

    print(
        "Expected parameters:",
        f"{capacity_parameter_counts[capacity_name]:,}"
    )


    # ------------------------------------------------------------
    # CLEAN MEMORY
    # ------------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()


    # ------------------------------------------------------------
    # RESET RANDOM STATE
    # ------------------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


    # ------------------------------------------------------------
    # DATASETS
    # ------------------------------------------------------------

    (
        training_dataset,
        validation_dataset,
        train_steps,
        validation_steps
    ) = build_datasets()


    print(
        "Training steps/epoch:",
        train_steps
    )

    print(
        "Validation steps:",
        validation_steps
    )


    # ------------------------------------------------------------
    # BUILD MODEL
    # ------------------------------------------------------------

    segmentation_model = (
        build_capacity_unet(
            filters=filters,
            input_shape=(
                SELECTED_RESOLUTION,
                SELECTED_RESOLUTION,
                1
            )
        )
    )


    actual_parameters = int(
        segmentation_model.count_params()
    )


    assert (
        actual_parameters
        ==
        capacity_parameter_counts[
            capacity_name
        ]
    )


    # ------------------------------------------------------------
    # OPTIMIZER
    # ------------------------------------------------------------

    optimizer_instance = (
        tf.keras.optimizers.Adam(
            learning_rate=
                SELECTED_LEARNING_RATE
        )
    )


    segmentation_model.compile(
        optimizer=optimizer_instance,
        loss=bce_dice_loss,
        metrics=[
            dice_coefficient,
            iou_metric
        ]
    )


    # ------------------------------------------------------------
    # OUTPUT PATHS
    # ------------------------------------------------------------

    model_path = (
        P1C_MODEL_DIR /
        f"{capacity_name}_best.keras"
    )

    history_path = (
        P1C_HISTORY_DIR /
        f"{capacity_name}_history.csv"
    )


    # ------------------------------------------------------------
    # CHECKPOINT
    # ------------------------------------------------------------

    checkpoint = (
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(
                model_path
            ),
            monitor=
                "val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    )


    # ------------------------------------------------------------
    # TRAIN
    # ------------------------------------------------------------

    start_time = time.time()


    history = segmentation_model.fit(
        training_dataset,
        validation_data=
            validation_dataset,
        epochs=MAX_EPOCHS,
        steps_per_epoch=
            train_steps,
        validation_steps=
            validation_steps,
        callbacks=[
            checkpoint
        ],
        verbose=1
    )


    training_minutes = (
        time.time() -
        start_time
    ) / 60.0


    # ------------------------------------------------------------
    # SAVE HISTORY
    # ------------------------------------------------------------

    history_df = pd.DataFrame(
        history.history
    )

    history_df.index = (
        np.arange(
            len(history_df)
        ) + 1
    )

    history_df.index.name = "epoch"

    history_df.to_csv(
        history_path
    )


    # ------------------------------------------------------------
    # BEST EPOCH
    # ------------------------------------------------------------

    val_dice_values = np.asarray(
        history.history[
            "val_dice_coefficient"
        ],
        dtype=float
    )

    best_index = int(
        np.argmax(
            val_dice_values
        )
    )


    result = {

        "capacity":
            capacity_name,

        "filters":
            str(filters),

        "parameters":
            actual_parameters,

        "resolution":
            SELECTED_RESOLUTION,

        "learning_rate":
            SELECTED_LEARNING_RATE,

        "optimizer":
            SELECTED_OPTIMIZER,

        "batch_size":
            SELECTED_BATCH_SIZE,

        "augmentation":
            False,

        "epochs":
            MAX_EPOCHS,

        "best_epoch":
            int(
                best_index + 1
            ),

        "best_val_dice_training":
            float(
                val_dice_values[
                    best_index
                ]
            ),

        "val_iou_at_best_dice_epoch":
            float(
                history.history[
                    "val_iou_metric"
                ][best_index]
            ),

        "training_minutes":
            float(
                training_minutes
            ),

        "model_path":
            str(model_path),

        "history_path":
            str(history_path)
    }


    print("\n" + "-" * 70)

    print(
        f"{capacity_name.upper()} COMPLETE"
    )

    print(
        "Parameters:",
        f"{actual_parameters:,}"
    )

    print(
        "Best epoch:",
        result["best_epoch"]
    )

    print(
        "Best soft validation Dice:",
        f"{result['best_val_dice_training']:.6f}"
    )

    print(
        "Validation IoU:",
        f"{result['val_iou_at_best_dice_epoch']:.6f}"
    )

    print(
        "Training time:",
        f"{training_minutes:.2f} minutes"
    )

    print(
        "Checkpoint:",
        model_path
    )

    print("-" * 70)


    # ------------------------------------------------------------
    # CLEAN MEMORY
    # ------------------------------------------------------------

    del segmentation_model
    del optimizer_instance
    del history
    del training_dataset
    del validation_dataset

    tf.keras.backend.clear_session()
    gc.collect()


    return result

In [ ]:
small_result = (
    train_capacity_experiment(
        "small"
    )
)

small_result


TRAINING CAPACITY: SMALL
Filters: [8, 16, 32, 64, 128]
Expected parameters: 485,673
Training steps/epoch: 400
Validation steps: 50
Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - dice_coefficient: 0.1867 - iou_metric: 0.1071 - loss: 1.3332
Epoch 1: val_dice_coefficient improved from None to 0.41965, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/small_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/small_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - dice_coefficient: 0.2753 - iou_metric: 0.1675 - loss: 1.0640 - val_dice_coefficient: 0.4197 - val_iou_metric: 0.2676 - val_loss: 0.7262
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - dice_coefficient: 0.4506 - iou_metric: 0.2938 - loss: 0.6999
Epoch 2: val_dice_coefficient improved from 0.41965 to 0.51976, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity

{'capacity': 'small',
 'filters': '[8, 16, 32, 64, 128]',
 'parameters': 485673,
 'resolution': 256,
 'learning_rate': 0.0001,
 'optimizer': 'Adam',
 'batch_size': 2,
 'augmentation': False,
 'epochs': 30,
 'best_epoch': 29,
 'best_val_dice_training': 0.9186065793037415,
 'val_iou_at_best_dice_epoch': 0.8524643778800964,
 'training_minutes': 11.393233076731365,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/small_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/histories/small_history.csv'}

In [ ]:
medium_result = (
    train_capacity_experiment(
        "medium"
    )
)

medium_result


TRAINING CAPACITY: MEDIUM
Filters: [16, 32, 64, 128, 256]
Expected parameters: 1,940,817
Training steps/epoch: 400
Validation steps: 50
Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - dice_coefficient: 0.2685 - iou_metric: 0.1694 - loss: 1.1392
Epoch 1: val_dice_coefficient improved from None to 0.70212, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/medium_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/medium_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 33s 61ms/step - dice_coefficient: 0.4339 - iou_metric: 0.3007 - loss: 0.8237 - val_dice_coefficient: 0.7021 - val_iou_metric: 0.5457 - val_loss: 0.4269
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - dice_coefficient: 0.7390 - iou_metric: 0.5959 - loss: 0.3813
Epoch 2: val_dice_coefficient improved from 0.70212 to 0.83733, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_c

{'capacity': 'medium',
 'filters': '[16, 32, 64, 128, 256]',
 'parameters': 1940817,
 'resolution': 256,
 'learning_rate': 0.0001,
 'optimizer': 'Adam',
 'batch_size': 2,
 'augmentation': False,
 'epochs': 30,
 'best_epoch': 30,
 'best_val_dice_training': 0.9289665222167969,
 'val_iou_at_best_dice_epoch': 0.8701490163803101,
 'training_minutes': 11.720148718357086,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/medium_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/histories/medium_history.csv'}

In [ ]:
large_result = (
    train_capacity_experiment(
        "large"
    )
)

large_result


TRAINING CAPACITY: LARGE
Filters: [32, 64, 128, 256, 512]
Expected parameters: 7,759,521
Training steps/epoch: 400
Validation steps: 50
Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - dice_coefficient: 0.3204 - iou_metric: 0.2122 - loss: 1.0459
Epoch 1: val_dice_coefficient improved from None to 0.82047, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/large_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/large_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 38s 82ms/step - dice_coefficient: 0.5133 - iou_metric: 0.3781 - loss: 0.7094 - val_dice_coefficient: 0.8205 - val_iou_metric: 0.7009 - val_loss: 0.2677
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - dice_coefficient: 0.8141 - iou_metric: 0.6952 - loss: 0.2720
Epoch 2: val_dice_coefficient improved from 0.82047 to 0.85203, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_cap

{'capacity': 'large',
 'filters': '[32, 64, 128, 256, 512]',
 'parameters': 7759521,
 'resolution': 256,
 'learning_rate': 0.0001,
 'optimizer': 'Adam',
 'batch_size': 2,
 'augmentation': False,
 'epochs': 30,
 'best_epoch': 28,
 'best_val_dice_training': 0.9335587024688721,
 'val_iou_at_best_dice_epoch': 0.8774716258049011,
 'training_minutes': 16.985555613040923,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/models/large_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/histories/large_history.csv'}

In [ ]:
# ================================================================
# PHASE 1C — CELL 19
# CAPACITY TRAINING SUMMARY
# ================================================================

capacity_training_summary = (
    pd.DataFrame([
        small_result,
        medium_result,
        large_result
    ])
)


CAPACITY_TRAINING_SUMMARY_PATH = (
    P1C_TABLE_DIR /
    "capacity_training_summary.csv"
)


capacity_training_summary.to_csv(
    CAPACITY_TRAINING_SUMMARY_PATH,
    index=False
)


display_columns = [

    "capacity",
    "parameters",
    "best_epoch",
    "best_val_dice_training",
    "val_iou_at_best_dice_epoch",
    "training_minutes"
]


display(
    capacity_training_summary[
        display_columns
    ].round(6)
)


print(
    "\n✓ Capacity training summary saved:"
)

print(
    CAPACITY_TRAINING_SUMMARY_PATH
)

,capacity,parameters,best_epoch,best_val_dice_training,val_iou_at_best_dice_epoch,training_minutes
0,small,485673,29,0.918607,0.852464,11.393233
1,medium,1940817,30,0.928967,0.870149,11.720149
2,large,7759521,28,0.933559,0.877472,16.985556



✓ Capacity training summary saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/tables/capacity_training_summary.csv


In [ ]:
# ================================================================
# PHASE 1C — CELL 20
# EXPERIMENT SANITY CHECK
# ================================================================

assert (
    int(
        capacity_training_summary.loc[
            capacity_training_summary[
                "capacity"
            ] == "small",
            "parameters"
        ].iloc[0]
    )
    ==
    485_673
)


assert (
    int(
        capacity_training_summary.loc[
            capacity_training_summary[
                "capacity"
            ] == "medium",
            "parameters"
        ].iloc[0]
    )
    ==
    1_940_817
)


assert (
    int(
        capacity_training_summary.loc[
            capacity_training_summary[
                "capacity"
            ] == "large",
            "parameters"
        ].iloc[0]
    )
    ==
    7_759_521
)


for capacity_name in [
    "small",
    "medium",
    "large"
]:

    checkpoint_path = (
        P1C_MODEL_DIR /
        f"{capacity_name}_best.keras"
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: "
        f"{checkpoint_path}"
    )


print("✓ Small checkpoint exists.")
print("✓ Medium checkpoint exists.")
print("✓ Large checkpoint exists.")

print("\nParameter progression:")

print(
    f"Small  → "
    f"{485_673:,}"
)

print(
    f"Medium → "
    f"{1_940_817:,}"
)

print(
    f"Large  → "
    f"{7_759_521:,}"
)

print("\n✓ Phase 1C training integrity checks passed.")
print("✓ Test set has still NOT been used.")

✓ Small checkpoint exists.
✓ Medium checkpoint exists.
✓ Large checkpoint exists.

Parameter progression:
Small  → 485,673
Medium → 1,940,817
Large  → 7,759,521

✓ Phase 1C training integrity checks passed.
✓ Test set has still NOT been used.


In [ ]:
# ================================================================
# PHASE 1C — CELL 21
# POSTPROCESSING + VALIDATION METRICS
# ================================================================

def keep_largest_component(mask):

    mask = np.asarray(mask).astype(np.uint8)

    if mask.sum() == 0:
        return np.zeros_like(mask, dtype=bool)

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    if n_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    largest_label = (
        1 +
        np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )
    )

    return labels == largest_label


def postprocess_prediction(mask):

    largest = keep_largest_component(mask)

    filled = binary_fill_holes(
        largest
    )

    return np.asarray(
        filled,
        dtype=bool
    )


def binary_dice(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    denominator = gt.sum() + pred.sum()

    if denominator == 0:
        return 1.0

    return (
        2.0 * intersection /
        denominator
    )


def binary_iou(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    union = np.logical_or(
        gt, pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def binary_precision(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(
        gt, pred
    ).sum()

    fp = np.logical_and(
        ~gt, pred
    ).sum()

    if (tp + fp) == 0:
        return (
            1.0
            if gt.sum() == 0
            else 0.0
        )

    return tp / (tp + fp)


def binary_recall(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(
        gt, pred
    ).sum()

    fn = np.logical_and(
        gt, ~pred
    ).sum()

    if (tp + fn) == 0:
        return 1.0

    return tp / (tp + fn)


def boundary_points(mask):

    mask = np.asarray(mask).astype(np.uint8)

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    eroded = cv2.erode(
        mask,
        kernel,
        iterations=1
    )

    boundary = (
        mask.astype(bool)
        &
        ~eroded.astype(bool)
    )

    return np.column_stack(
        np.where(boundary)
    )


def hd95_pixels(gt, pred):

    gt_points = boundary_points(gt)
    pred_points = boundary_points(pred)

    if (
        len(gt_points) == 0
        or len(pred_points) == 0
    ):
        return np.nan

    distances = cdist(
        gt_points,
        pred_points
    )

    gt_to_pred = distances.min(
        axis=1
    )

    pred_to_gt = distances.min(
        axis=0
    )

    all_distances = np.concatenate([
        gt_to_pred,
        pred_to_gt
    ])

    return float(
        np.percentile(
            all_distances,
            95
        )
    )


print("✓ Postprocessing and evaluation metrics ready.")

✓ Postprocessing and evaluation metrics ready.


In [ ]:
# ================================================================
# PHASE 1C — CELL 22
# CAPACITY CHECKPOINT EVALUATOR
# ================================================================

def evaluate_capacity_checkpoint(
    capacity_name
):

    assert capacity_name in [
        "small",
        "medium",
        "large"
    ]

    model_path = (
        P1C_MODEL_DIR /
        f"{capacity_name}_best.keras"
    )

    assert model_path.exists(), (
        f"Checkpoint missing:\n{model_path}"
    )


    print("\n" + "=" * 70)
    print(
        f"EVALUATING CAPACITY: "
        f"{capacity_name.upper()}"
    )
    print("=" * 70)


    tf.keras.backend.clear_session()
    gc.collect()


    evaluation_model = (
        tf.keras.models.load_model(
            model_path,
            custom_objects={
                "bce_dice_loss":
                    bce_dice_loss,

                "dice_coefficient":
                    dice_coefficient,

                "iou_metric":
                    iou_metric
            }
        )
    )


    actual_parameters = int(
        evaluation_model.count_params()
    )

    assert (
        actual_parameters
        ==
        capacity_parameter_counts[
            capacity_name
        ]
    )


    records = []


    for patient_number, pid in enumerate(
        val_ids,
        start=1
    ):

        for phase in ["ED", "ES"]:

            image, gt_mask = (
                load_camus_frame(
                    pid,
                    phase
                )
            )


            x, gt_resized = (
                preprocess_frame(
                    image,
                    gt_mask
                )
            )


            probability = (
                evaluation_model(
                    x[None, ...],
                    training=False
                )
                .numpy()[0, :, :, 0]
            )


            pred_raw = (
                probability >= THRESHOLD
            )


            pred_mask = (
                postprocess_prediction(
                    pred_raw
                )
            )


            gt_binary = (
                gt_resized[:, :, 0]
                > 0.5
            )


            gt_area = int(
                gt_binary.sum()
            )

            pred_area = int(
                pred_mask.sum()
            )


            if gt_area > 0:

                signed_area_error_pct = (
                    (pred_area - gt_area)
                    / gt_area
                    * 100.0
                )

                abs_area_error_pct = abs(
                    signed_area_error_pct
                )

            else:

                signed_area_error_pct = np.nan
                abs_area_error_pct = np.nan


            hd95 = hd95_pixels(
                gt_binary,
                pred_mask
            )


            records.append({

                "patient_id":
                    pid,

                "phase":
                    phase,

                "capacity":
                    capacity_name,

                "parameters":
                    actual_parameters,

                "dice":
                    binary_dice(
                        gt_binary,
                        pred_mask
                    ),

                "iou":
                    binary_iou(
                        gt_binary,
                        pred_mask
                    ),

                "precision":
                    binary_precision(
                        gt_binary,
                        pred_mask
                    ),

                "recall":
                    binary_recall(
                        gt_binary,
                        pred_mask
                    ),

                "hd95_pixels":
                    hd95,

                "normalized_hd95":
                    (
                        hd95 /
                        SELECTED_RESOLUTION
                        if np.isfinite(hd95)
                        else np.nan
                    ),

                "gt_area_pixels":
                    gt_area,

                "pred_area_pixels":
                    pred_area,

                "signed_area_error_pct":
                    signed_area_error_pct,

                "abs_area_error_pct":
                    abs_area_error_pct
            })


        if patient_number % 10 == 0:

            print(
                f"Evaluated "
                f"{patient_number}/"
                f"{len(val_ids)} patients"
            )


    results_df = pd.DataFrame(
        records
    )

    assert len(results_df) == 100


    output_path = (
        P1C_TABLE_DIR /
        f"{capacity_name}_validation_metrics.csv"
    )


    results_df.to_csv(
        output_path,
        index=False
    )


    print(
        "\nFrames evaluated:",
        len(results_df)
    )

    print(
        "Parameters:",
        f"{actual_parameters:,}"
    )

    print("Saved:")
    print(output_path)


    del evaluation_model

    tf.keras.backend.clear_session()
    gc.collect()


    return results_df

In [ ]:
# ================================================================
# PHASE 1C — CELL 23
# EVALUATE ALL CAPACITIES
# ================================================================

small_eval = (
    evaluate_capacity_checkpoint(
        "small"
    )
)


medium_eval = (
    evaluate_capacity_checkpoint(
        "medium"
    )
)


large_eval = (
    evaluate_capacity_checkpoint(
        "large"
    )
)


assert len(small_eval) == 100
assert len(medium_eval) == 100
assert len(large_eval) == 100


print(
    "\n✓ All three capacity checkpoints evaluated."
)

print(
    "✓ Same 100 validation frames used for every model."
)

print(
    "✓ Test set was NOT used."
)


EVALUATING CAPACITY: SMALL
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Parameters: 485,673
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/tables/small_validation_metrics.csv

EVALUATING CAPACITY: MEDIUM
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Parameters: 1,940,817
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/tables/medium_validation_metrics.csv

EVALUATING CAPACITY: LARGE
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Parameters: 7,759,521
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/tables/large_validation_metrics.csv

✓ All three capacity checkpoints evaluated.
✓ Same 100 valid

In [ ]:
# ================================================================
# PHASE 1C — CELL 24
# FINAL CAPACITY VALIDATION COMPARISON
# ================================================================

def summarize_capacity_results(
    evaluation_df,
    training_result
):

    ed = evaluation_df[
        evaluation_df["phase"] == "ED"
    ]

    es = evaluation_df[
        evaluation_df["phase"] == "ES"
    ]


    return {

        "capacity":
            training_result[
                "capacity"
            ],

        "parameters":
            training_result[
                "parameters"
            ],

        "best_epoch":
            training_result[
                "best_epoch"
            ],

        "training_minutes":
            training_result[
                "training_minutes"
            ],

        "overall_dice":
            evaluation_df[
                "dice"
            ].mean(),

        "ED_dice":
            ed[
                "dice"
            ].mean(),

        "ES_dice":
            es[
                "dice"
            ].mean(),

        "overall_iou":
            evaluation_df[
                "iou"
            ].mean(),

        "precision":
            evaluation_df[
                "precision"
            ].mean(),

        "recall":
            evaluation_df[
                "recall"
            ].mean(),

        "normalized_hd95":
            evaluation_df[
                "normalized_hd95"
            ].mean(),

        "ED_area_MAE_pct":
            ed[
                "abs_area_error_pct"
            ].mean(),

        "ES_area_MAE_pct":
            es[
                "abs_area_error_pct"
            ].mean(),

        "overall_area_MAE_pct":
            evaluation_df[
                "abs_area_error_pct"
            ].mean(),

        "ED_signed_area_error_pct":
            ed[
                "signed_area_error_pct"
            ].mean(),

        "ES_signed_area_error_pct":
            es[
                "signed_area_error_pct"
            ].mean(),

        "overall_signed_area_error_pct":
            evaluation_df[
                "signed_area_error_pct"
            ].mean()
    }


capacity_comparison = pd.DataFrame([

    summarize_capacity_results(
        small_eval,
        small_result
    ),

    summarize_capacity_results(
        medium_eval,
        medium_result
    ),

    summarize_capacity_results(
        large_eval,
        large_result
    )

])


CAPACITY_COMPARISON_PATH = (
    P1C_TABLE_DIR /
    "capacity_validation_comparison.csv"
)


capacity_comparison.to_csv(
    CAPACITY_COMPARISON_PATH,
    index=False
)


display(
    capacity_comparison.round(4)
)


print(
    "\n✓ Final capacity comparison saved:"
)

print(
    CAPACITY_COMPARISON_PATH
)

,capacity,parameters,best_epoch,training_minutes,overall_dice,ED_dice,ES_dice,overall_iou,precision,recall,normalized_hd95,ED_area_MAE_pct,ES_area_MAE_pct,overall_area_MAE_pct,ED_signed_area_error_pct,ES_signed_area_error_pct,overall_signed_area_error_pct
0,small,485673,29,11.3932,0.9248,0.9377,0.9118,0.8632,0.9328,0.9238,0.0256,6.8932,11.3736,9.1334,-1.0807,1.2444,0.0818
1,medium,1940817,30,11.7201,0.9304,0.9404,0.9205,0.8734,0.9208,0.9463,0.0240,7.1633,10.5272,8.8452,2.6587,4.4911,3.5749
2,large,7759521,28,16.9856,0.9350,0.9443,0.9257,0.8801,0.9395,0.9356,0.0217,5.8389,9.5696,7.7042,-0.9072,1.6548,0.3738



✓ Final capacity comparison saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/tables/capacity_validation_comparison.csv


In [ ]:
# ================================================================
# PHASE 1C — CELL 25
# LARGE vs MEDIUM — PRACTICAL IMPROVEMENT
# ================================================================

medium_row = (
    capacity_comparison[
        capacity_comparison[
            "capacity"
        ] == "medium"
    ].iloc[0]
)


large_row = (
    capacity_comparison[
        capacity_comparison[
            "capacity"
        ] == "large"
    ].iloc[0]
)


capacity_gain = pd.DataFrame({

    "metric": [

        "Parameters",
        "Training minutes",

        "Overall Dice",
        "ED Dice",
        "ES Dice",
        "Overall IoU",

        "Normalized HD95",

        "ED Area MAE %",
        "ES Area MAE %",
        "Overall Area MAE %"
    ],

    "medium": [

        medium_row["parameters"],
        medium_row["training_minutes"],

        medium_row["overall_dice"],
        medium_row["ED_dice"],
        medium_row["ES_dice"],
        medium_row["overall_iou"],

        medium_row["normalized_hd95"],

        medium_row["ED_area_MAE_pct"],
        medium_row["ES_area_MAE_pct"],
        medium_row["overall_area_MAE_pct"]
    ],

    "large": [

        large_row["parameters"],
        large_row["training_minutes"],

        large_row["overall_dice"],
        large_row["ED_dice"],
        large_row["ES_dice"],
        large_row["overall_iou"],

        large_row["normalized_hd95"],

        large_row["ED_area_MAE_pct"],
        large_row["ES_area_MAE_pct"],
        large_row["overall_area_MAE_pct"]
    ]
})


capacity_gain[
    "large_minus_medium"
] = (
    capacity_gain["large"]
    -
    capacity_gain["medium"]
)


display(
    capacity_gain.round(6)
)


print(
    "\nParameter multiplier:"
)

print(
    f"{large_row['parameters'] / medium_row['parameters']:.2f}×"
)


print(
    "\nTraining-time multiplier:"
)

print(
    f"{large_row['training_minutes'] / medium_row['training_minutes']:.2f}×"
)

,metric,medium,large,large_minus_medium
0,Parameters,1.940817e+06,7.759521e+06,5.818704e+06
1,Training minutes,1.172015e+01,1.698556e+01,5.265407e+00
2,Overall Dice,9.304200e-01,9.350130e-01,4.593000e-03
3,ED Dice,9.403780e-01,9.442840e-01,3.906000e-03
4,ES Dice,9.204610e-01,9.257410e-01,5.280000e-03
5,Overall IoU,8.734220e-01,8.800760e-01,6.654000e-03
6,Normalized HD95,2.399300e-02,2.165600e-02,-2.337000e-03
7,ED Area MAE %,7.163292e+00,5.838892e+00,-1.324400e+00
8,ES Area MAE %,1.052720e+01,9.569602e+00,-9.576030e-01
9,Overall Area MAE %,8.845248e+00,7.704247e+00,-1.141001e+00



Parameter multiplier:
4.00×

Training-time multiplier:
1.45×


In [ ]:
# ================================================================
# PHASE 1C — CELL 26
# PAIRED LARGE vs MEDIUM FRAME-LEVEL COMPARISON
# ================================================================

assert (
    medium_eval[
        ["patient_id", "phase"]
    ]
    .reset_index(drop=True)
    .equals(
        large_eval[
            ["patient_id", "phase"]
        ]
        .reset_index(drop=True)
    )
)


paired_capacity = pd.DataFrame({

    "patient_id":
        medium_eval[
            "patient_id"
        ].values,

    "phase":
        medium_eval[
            "phase"
        ].values,

    "dice_change_large_minus_medium":
        (
            large_eval["dice"].values
            -
            medium_eval["dice"].values
        ),

    "iou_change_large_minus_medium":
        (
            large_eval["iou"].values
            -
            medium_eval["iou"].values
        ),

    "hd95_change_large_minus_medium":
        (
            large_eval[
                "normalized_hd95"
            ].values
            -
            medium_eval[
                "normalized_hd95"
            ].values
        ),

    "area_MAE_change_large_minus_medium":
        (
            large_eval[
                "abs_area_error_pct"
            ].values
            -
            medium_eval[
                "abs_area_error_pct"
            ].values
        )
})


PAIRED_CAPACITY_PATH = (
    P1C_TABLE_DIR /
    "large_vs_medium_paired_metrics.csv"
)


paired_capacity.to_csv(
    PAIRED_CAPACITY_PATH,
    index=False
)


paired_capacity_summary = pd.DataFrame({

    "metric": [
        "Dice",
        "IoU",
        "Normalized HD95",
        "Area MAE %"
    ],

    "mean_change_large_minus_medium": [

        paired_capacity[
            "dice_change_large_minus_medium"
        ].mean(),

        paired_capacity[
            "iou_change_large_minus_medium"
        ].mean(),

        paired_capacity[
            "hd95_change_large_minus_medium"
        ].mean(),

        paired_capacity[
            "area_MAE_change_large_minus_medium"
        ].mean()
    ]
})


display(
    paired_capacity_summary.round(6)
)


dice_improved_large = int(
    (
        paired_capacity[
            "dice_change_large_minus_medium"
        ] > 0
    ).sum()
)


dice_worsened_large = int(
    (
        paired_capacity[
            "dice_change_large_minus_medium"
        ] < 0
    ).sum()
)


area_improved_large = int(
    (
        paired_capacity[
            "area_MAE_change_large_minus_medium"
        ] < 0
    ).sum()
)


print(
    f"\nFrames where Large improved Dice: "
    f"{dice_improved_large}/100"
)

print(
    f"Frames where Large worsened Dice: "
    f"{dice_worsened_large}/100"
)

print(
    f"Frames where Large reduced area error: "
    f"{area_improved_large}/100"
)

print("\nSaved:")
print(PAIRED_CAPACITY_PATH)

,metric,mean_change_large_minus_medium
0,Dice,0.004593
1,IoU,0.006654
2,Normalized HD95,-0.002337
3,Area MAE %,-1.141001



Frames where Large improved Dice: 54/100
Frames where Large worsened Dice: 46/100
Frames where Large reduced area error: 58/100

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/tables/large_vs_medium_paired_metrics.csv


In [ ]:
# ================================================================
# PHASE 1C — CELL 27
# FINAL MODEL CAPACITY DECISION
# ================================================================

SELECTED_CAPACITY = "large"
SELECTED_FILTERS = CAPACITY_CONFIGS[SELECTED_CAPACITY]
SELECTED_PARAMETER_COUNT = capacity_parameter_counts[SELECTED_CAPACITY]

selected_row = (
    capacity_comparison[
        capacity_comparison["capacity"] == SELECTED_CAPACITY
    ].iloc[0]
)

medium_row = (
    capacity_comparison[
        capacity_comparison["capacity"] == "medium"
    ].iloc[0]
)


capacity_decision = {

    "experiment":
        "Phase 1C — Controlled Model Capacity Experiment",

    "status":
        "COMPLETED",

    "selected_capacity":
        SELECTED_CAPACITY,

    "selected_filters":
        SELECTED_FILTERS,

    "selected_parameter_count":
        int(SELECTED_PARAMETER_COUNT),

    "fixed_settings": {
        "resolution": SELECTED_RESOLUTION,
        "learning_rate": SELECTED_LEARNING_RATE,
        "optimizer": SELECTED_OPTIMIZER,
        "batch_size": SELECTED_BATCH_SIZE,
        "augmentation": False,
        "loss": "BCE + Dice",
        "epochs": MAX_EPOCHS,
        "threshold": THRESHOLD,
        "seed": SEED
    },

    "capacities_compared": {
        name: {
            "filters": filters,
            "parameters": int(capacity_parameter_counts[name])
        }
        for name, filters in CAPACITY_CONFIGS.items()
    },

    "selected_validation_results": {
        "overall_dice":
            float(selected_row["overall_dice"]),

        "ED_dice":
            float(selected_row["ED_dice"]),

        "ES_dice":
            float(selected_row["ES_dice"]),

        "overall_iou":
            float(selected_row["overall_iou"]),

        "precision":
            float(selected_row["precision"]),

        "recall":
            float(selected_row["recall"]),

        "normalized_hd95":
            float(selected_row["normalized_hd95"]),

        "ED_area_MAE_pct":
            float(selected_row["ED_area_MAE_pct"]),

        "ES_area_MAE_pct":
            float(selected_row["ES_area_MAE_pct"]),

        "overall_area_MAE_pct":
            float(selected_row["overall_area_MAE_pct"]),

        "ED_signed_area_error_pct":
            float(selected_row["ED_signed_area_error_pct"]),

        "ES_signed_area_error_pct":
            float(selected_row["ES_signed_area_error_pct"]),

        "overall_signed_area_error_pct":
            float(selected_row["overall_signed_area_error_pct"])
    },

    "large_vs_medium": {
        "parameter_multiplier":
            float(
                selected_row["parameters"] /
                medium_row["parameters"]
            ),

        "overall_dice_change":
            float(
                selected_row["overall_dice"] -
                medium_row["overall_dice"]
            ),

        "ED_dice_change":
            float(
                selected_row["ED_dice"] -
                medium_row["ED_dice"]
            ),

        "ES_dice_change":
            float(
                selected_row["ES_dice"] -
                medium_row["ES_dice"]
            ),

        "normalized_hd95_change":
            float(
                selected_row["normalized_hd95"] -
                medium_row["normalized_hd95"]
            ),

        "overall_area_MAE_change_pct":
            float(
                selected_row["overall_area_MAE_pct"] -
                medium_row["overall_area_MAE_pct"]
            ),

        "ES_area_MAE_change_pct":
            float(
                selected_row["ES_area_MAE_pct"] -
                medium_row["ES_area_MAE_pct"]
            )
    },

    "paired_large_vs_medium": {
        "mean_dice_change":
            float(
                paired_capacity[
                    "dice_change_large_minus_medium"
                ].mean()
            ),

        "mean_iou_change":
            float(
                paired_capacity[
                    "iou_change_large_minus_medium"
                ].mean()
            ),

        "mean_normalized_hd95_change":
            float(
                paired_capacity[
                    "hd95_change_large_minus_medium"
                ].mean()
            ),

        "mean_area_MAE_change_pct":
            float(
                paired_capacity[
                    "area_MAE_change_large_minus_medium"
                ].mean()
            ),

        "frames_dice_improved":
            int(dice_improved_large),

        "frames_dice_worsened":
            int(dice_worsened_large),

        "frames_area_error_reduced":
            int(area_improved_large)
    },

    "selection_dataset":
        "Validation set only",

    "test_set_used":
        False,

    "selection_reason": (
        "The Large U-Net was selected because it achieved the strongest "
        "validation performance across segmentation overlap, boundary "
        "accuracy, and LV area fidelity. Compared with the Medium U-Net, "
        "it improved overall, ED, and ES Dice and IoU, reduced normalized "
        "HD95, and reduced ED, ES, and overall area MAE. The improvement "
        "in ES Dice and ES area error is particularly relevant for the "
        "subsequent quantitative LV analysis."
    ),

    "interpretation_note": (
        "The selected Large architecture contains approximately four "
        "times as many parameters as the Medium architecture. Capacity "
        "selection is based on validation performance under the fixed "
        "Phase 1B training protocol; the test set was not used."
    ),

    "next_experiment":
        "Phase 1D — Controlled Loss Function Experiment"
}


CAPACITY_DECISION_PATH = (
    P1C_CONFIG_DIR /
    "phase1C_capacity_decision.json"
)


with open(CAPACITY_DECISION_PATH, "w") as f:
    json.dump(
        capacity_decision,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1C COMPLETED")
print("=" * 70)

print("\nSELECTED CAPACITY:")
print("Large U-Net")

print("\nArchitecture:")
print(SELECTED_FILTERS)

print(
    f"Parameters: "
    f"{SELECTED_PARAMETER_COUNT:,}"
)

print("\nVALIDATION RESULTS")
print(
    f"Overall Dice     : "
    f"{selected_row['overall_dice']:.4f}"
)
print(
    f"ED Dice          : "
    f"{selected_row['ED_dice']:.4f}"
)
print(
    f"ES Dice          : "
    f"{selected_row['ES_dice']:.4f}"
)
print(
    f"Overall IoU      : "
    f"{selected_row['overall_iou']:.4f}"
)
print(
    f"Normalized HD95  : "
    f"{selected_row['normalized_hd95']:.4f}"
)
print(
    f"ED Area MAE      : "
    f"{selected_row['ED_area_MAE_pct']:.2f}%"
)
print(
    f"ES Area MAE      : "
    f"{selected_row['ES_area_MAE_pct']:.2f}%"
)
print(
    f"Overall Area MAE : "
    f"{selected_row['overall_area_MAE_pct']:.2f}%"
)

print("\nTEST SET USED: NO")

print("\nCURRENT LOCKED PIPELINE")
print("Resolution   : 256×256")
print("Learning rate: 1e-4")
print("Optimizer    : Adam")
print("Batch size   : 2")
print("Augmentation : None")
print("Capacity     : Large")
print(f"Parameters   : {SELECTED_PARAMETER_COUNT:,}")
print("Loss         : BCE + Dice  [NEXT VARIABLE]")

print("\nDecision saved:")
print(CAPACITY_DECISION_PATH)

print("\nNEXT:")
print("Phase 1D — Controlled Loss Function Experiment")

print("\n✓ Phase 1C is now complete.")

PHASE 1C COMPLETED

SELECTED CAPACITY:
Large U-Net

Architecture:
[32, 64, 128, 256, 512]
Parameters: 7,759,521

VALIDATION RESULTS
Overall Dice     : 0.9350
ED Dice          : 0.9443
ES Dice          : 0.9257
Overall IoU      : 0.8801
Normalized HD95  : 0.0217
ED Area MAE      : 5.84%
ES Area MAE      : 9.57%
Overall Area MAE : 7.70%

TEST SET USED: NO

CURRENT LOCKED PIPELINE
Resolution   : 256×256
Learning rate: 1e-4
Optimizer    : Adam
Batch size   : 2
Augmentation : None
Capacity     : Large
Parameters   : 7,759,521
Loss         : BCE + Dice  [NEXT VARIABLE]

Decision saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1C_capacity/configs/phase1C_capacity_decision.json

NEXT:
Phase 1D — Controlled Loss Function Experiment

✓ Phase 1C is now complete.
